# 10 Non-IID, Dropout, Straggler, and Scalability Study

Runs the registered one-factor-at-a-time and hardest-joint stress matrix for matched FedAvg and EcoTrust protocols. Every run is checkpointed by dataset, scenario, method, seed, and round.

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

EXPECTED_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')

def valid_root(path):
    return (path / 'configs' / 'project_colab.json').is_file() and (path / 'src' / 'ecotrust').is_dir()

try:
    from google.colab import drive  # type: ignore
    if not valid_root(EXPECTED_ROOT):
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        subprocess.run(['fusermount', '-uz', '/content/drive'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        shutil.rmtree('/content/drive', ignore_errors=True)
        Path('/content/drive').mkdir(parents=True, exist_ok=True)
        drive.mount('/content/drive', force_remount=False, timeout_ms=120000)
except ImportError:
    pass

PROJECT_ROOT = EXPECTED_ROOT
if not valid_root(PROJECT_ROOT):
    raise RuntimeError(f'Canonical Drive project is unavailable: {PROJECT_ROOT}')
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'publication_experiments.json'
RUN_MODE = 'full'
DEVICE = 'cpu'
RESUME = True
SHARD_INDEX = int(os.environ.get('ECOTRUST_SHARD_INDEX', '0'))
SHARD_COUNT = int(os.environ.get('ECOTRUST_SHARD_COUNT', '1'))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

importlib.invalidate_caches()
import ecotrust.evaluation.publication_stage10 as stage10
importlib.reload(stage10)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)
print('REGISTRY_PATH:', REGISTRY_PATH)
print('RUN_MODE:', RUN_MODE)
print('DEVICE:', DEVICE)
print('RESUME:', RESUME)
print('SHARD:', f'{SHARD_INDEX + 1}/{SHARD_COUNT}')
print('STAGE10_SOURCE:', stage10.__file__)

result = stage10.run_stage10(
    CONFIG_PATH,
    REGISTRY_PATH,
    run_mode=RUN_MODE,
    device=DEVICE,
    resume=RESUME,
    shard_index=SHARD_INDEX,
    shard_count=SHARD_COUNT,
)
print('STAGE_10_STATUS:', result['status'])
print('STAGE_10_MANIFEST:', result['manifest'])
print('STAGE_10_RUNS:', len(result['runs']))
